In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Current working directory:", Path.cwd())

Project root: /home/ubuntu/renewable-energy-forecasting-pipeline
Current working directory: /home/ubuntu/renewable-energy-forecasting-pipeline


In [2]:
from dotenv import load_dotenv
load_dotenv()

import os
print(os.environ.get("PROJECT_USER_CONFIG"))

configs/users/syed.yaml


In [3]:
from pyspark.sql import functions as F
from src.common.spark_utils import get_spark_session
from src.common.paths import Paths

spark = get_spark_session(app_name="05_scaled_etl_validation")
paths = Paths()

print("Bronze path:", paths.bronze_isd)
print("Silver path:", paths.silver_weather)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ubuntu/.ivy2/cache
The jars for the packages stored in: /home/ubuntu/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b5a9bfe8-a726-4d92-ad22-1c02ab883c72;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 450ms :: artifacts dl 28ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	--------------------------------

Bronze path: s3a://syed-datsbd-s2026/bronze/isd
Silver path: s3a://syed-datsbd-s2026/silver/weather


In [5]:
bronze_df = spark.read.parquet(paths.bronze_isd)
silver_df = spark.read.parquet(paths.silver_weather)

print("Bronze loaded")
print("Silver loaded")

26/04/25 03:32:02 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Bronze loaded
Silver loaded


In [6]:
bronze_df.printSchema()

root
 |-- STATION: string (nullable = true)
 |-- DATE: string (nullable = true)
 |-- SOURCE: string (nullable = true)
 |-- LATITUDE: string (nullable = true)
 |-- LONGITUDE: string (nullable = true)
 |-- ELEVATION: string (nullable = true)
 |-- NAME: string (nullable = true)
 |-- REPORT_TYPE: string (nullable = true)
 |-- CALL_SIGN: string (nullable = true)
 |-- QUALITY_CONTROL: string (nullable = true)
 |-- WND: string (nullable = true)
 |-- CIG: string (nullable = true)
 |-- VIS: string (nullable = true)
 |-- TMP: string (nullable = true)
 |-- DEW: string (nullable = true)
 |-- SLP: string (nullable = true)
 |-- AA1: string (nullable = true)
 |-- AB1: string (nullable = true)
 |-- AE1: string (nullable = true)
 |-- AO1: string (nullable = true)
 |-- CB1: string (nullable = true)
 |-- CF1: string (nullable = true)
 |-- CF2: string (nullable = true)
 |-- CF3: string (nullable = true)
 |-- CG1: string (nullable = true)
 |-- CG2: string (nullable = true)
 |-- CG3: string (nullable = true

In [7]:
silver_df.printSchema()

root
 |-- station_id: string (nullable = true)
 |-- timestamp_utc: timestamp (nullable = true)
 |-- date_utc: date (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- wind_speed_ms: double (nullable = true)
 |-- wind_direction_degrees: integer (nullable = true)
 |-- temperature_c: decimal(17,6) (nullable = true)
 |-- dew_point_c: decimal(17,6) (nullable = true)
 |-- sea_level_pressure_hpa: decimal(17,6) (nullable = true)
 |-- visibility_distance_m: decimal(13,1) (nullable = true)
 |-- ceiling_height_m: decimal(13,1) (nullable = true)
 |-- STATION: string (nullable = true)
 |-- DATE: string (nullable = true)
 |-- SOURCE: string (nullable = true)
 |-- LATITUDE: string (nullable = true)
 |-- LONGITUDE: string (nullable = true)
 |-- ELEVATION: string (nullable = true)
 |-- NAME: string (nullable = true)
 |-- REPORT_TYPE: string (nullable = true)
 |-- CALL_SIGN: string (nullable = true)
 |-- QUALITY_CONTROL: s

In [8]:
bronze_count = bronze_df.count()
silver_count = silver_df.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Rows removed during parsing/cleaning/QC:", bronze_count - silver_count)
print("Silver retention rate:", silver_count / bronze_count if bronze_count else None)

Bronze rows: 35504907
Silver rows: 28893512
Rows removed during parsing/cleaning/QC: 6611395
Silver retention rate: 0.8137892601718405


In [9]:
bronze_df.groupBy("year").count().orderBy("year").show(truncate=False)

+----+--------+
|year|count   |
+----+--------+
|2018|11696219|
|2019|12094555|
|2020|11714133|
+----+--------+



In [10]:
silver_df.groupBy("year", "state").count().orderBy("year", "state").show(100, truncate=False)

+----+-----+-------+
|year|state|count  |
+----+-----+-------+
|2018|CA   |2002256|
|2018|FL   |1564053|
|2018|MN   |2256236|
|2018|TX   |3675645|
|2019|CA   |2044267|
|2019|FL   |1568955|
|2019|MN   |2571106|
|2019|TX   |3659929|
|2020|CA   |1979398|
|2020|FL   |1603829|
|2020|MN   |2399680|
|2020|TX   |3568158|
+----+-----+-------+



In [11]:
silver_df.groupBy("state").count().orderBy("state").show(truncate=False)

+-----+--------+
|state|count   |
+-----+--------+
|CA   |6025921 |
|FL   |4736837 |
|MN   |7227022 |
|TX   |10903732|
+-----+--------+



In [12]:
required_columns = [
    "station_id",
    "timestamp_utc",
    "date_utc",
    "year",
    "month",
    "day",
    "hour",
    "state",
    "region",
    "wind_speed_ms",
    "wind_direction_degrees",
    "temperature_c",
    "dew_point_c",
    "sea_level_pressure_hpa",
    "visibility_distance_m",
    "ceiling_height_m",
]

missing_columns = [c for c in required_columns if c not in silver_df.columns]

print("Missing required columns:", missing_columns)
print("Required columns present:", len(missing_columns) == 0)

Missing required columns: []
Required columns present: True


In [13]:
null_checks = [
    F.sum(F.col(c).isNull().cast("int")).alias(f"{c}_nulls")
    for c in required_columns
    if c in silver_df.columns
]

silver_df.select(*null_checks).show(truncate=False)

+----------------+-------------------+--------------+----------+-----------+---------+----------+-----------+------------+-------------------+----------------------------+-------------------+-----------------+----------------------------+---------------------------+----------------------+
|station_id_nulls|timestamp_utc_nulls|date_utc_nulls|year_nulls|month_nulls|day_nulls|hour_nulls|state_nulls|region_nulls|wind_speed_ms_nulls|wind_direction_degrees_nulls|temperature_c_nulls|dew_point_c_nulls|sea_level_pressure_hpa_nulls|visibility_distance_m_nulls|ceiling_height_m_nulls|
+----------------+-------------------+--------------+----------+-----------+---------+----------+-----------+------------+-------------------+----------------------------+-------------------+-----------------+----------------------------+---------------------------+----------------------+
|0               |0                  |0             |0         |0          |0        |0         |0          |0           |0       

In [14]:
silver_df.select(
    F.count("*").alias("rows"),
    F.count("wind_speed_ms").alias("non_null_wind"),
    F.sum((F.col("wind_speed_ms") == 0).cast("int")).alias("zero_wind_rows"),
    F.min("wind_speed_ms").alias("min_wind"),
    F.expr("percentile_approx(wind_speed_ms, 0.25)").alias("p25_wind"),
    F.expr("percentile_approx(wind_speed_ms, 0.5)").alias("median_wind"),
    F.expr("percentile_approx(wind_speed_ms, 0.75)").alias("p75_wind"),
    F.expr("percentile_approx(wind_speed_ms, 0.95)").alias("p95_wind"),
    F.max("wind_speed_ms").alias("max_wind"),
).show(truncate=False)

+--------+-------------+--------------+--------+--------+-----------+--------+--------+--------+
|rows    |non_null_wind|zero_wind_rows|min_wind|p25_wind|median_wind|p75_wind|p95_wind|max_wind|
+--------+-------------+--------------+--------+--------+-----------+--------+--------+--------+
|28893512|28893512     |5948808       |0.0     |1.5     |3.1        |4.6     |7.7     |61.3    |
+--------+-------------+--------------+--------+--------+-----------+--------+--------+--------+



In [15]:
plausibility = silver_df.select(
    F.sum((F.col("wind_speed_ms") < 0).cast("int")).alias("negative_wind_rows"),
    F.sum((F.col("wind_speed_ms") > 75).cast("int")).alias("wind_over_75_ms_rows"),
    F.sum((F.col("wind_direction_degrees") < 0).cast("int")).alias("negative_direction_rows"),
    F.sum((F.col("wind_direction_degrees") > 360).cast("int")).alias("direction_over_360_rows"),
    F.sum((F.col("temperature_c") < -90).cast("int")).alias("temperature_below_minus_90_rows"),
    F.sum((F.col("temperature_c") > 60).cast("int")).alias("temperature_above_60_rows"),
    F.sum((F.col("sea_level_pressure_hpa") < 800).cast("int")).alias("pressure_below_800_rows"),
    F.sum((F.col("sea_level_pressure_hpa") > 1100).cast("int")).alias("pressure_above_1100_rows"),
)

plausibility.show(truncate=False)

+------------------+--------------------+-----------------------+-----------------------+-------------------------------+-------------------------+-----------------------+------------------------+
|negative_wind_rows|wind_over_75_ms_rows|negative_direction_rows|direction_over_360_rows|temperature_below_minus_90_rows|temperature_above_60_rows|pressure_below_800_rows|pressure_above_1100_rows|
+------------------+--------------------+-----------------------+-----------------------+-------------------------------+-------------------------+-----------------------+------------------------+
|0                 |0                   |0                      |0                      |0                              |0                        |0                      |0                       |
+------------------+--------------------+-----------------------+-----------------------+-------------------------------+-------------------------+-----------------------+------------------------+



In [16]:
silver_df.select(
    F.countDistinct("station_id").alias("distinct_stations"),
    F.countDistinct("state").alias("distinct_states"),
    F.countDistinct("year").alias("distinct_years"),
).show(truncate=False)

silver_df.groupBy("state").agg(
    F.countDistinct("station_id").alias("stations")
).orderBy("state").show(truncate=False)

+-----------------+---------------+--------------+
|distinct_stations|distinct_states|distinct_years|
+-----------------+---------------+--------------+
|626              |4              |3             |
+-----------------+---------------+--------------+



+-----+--------+
|state|stations|
+-----+--------+
|CA   |169     |
|FL   |140     |
|MN   |100     |
|TX   |217     |
+-----+--------+



In [17]:
import time

start = time.time()

tx_2020_count = (
    silver_df
    .filter((F.col("year") == 2020) & (F.col("state") == "TX"))
    .count()
)

elapsed = time.time() - start

print("Rows for year=2020/state=TX:", tx_2020_count)
print("Read/filter/count time seconds:", round(elapsed, 2))

Rows for year=2020/state=TX: 3568158
Read/filter/count time seconds: 1.46


In [18]:
silver_df.select(
    "station_id",
    "timestamp_utc",
    "year",
    "month",
    "day",
    "hour",
    "state",
    "region",
    "wind_speed_ms",
    "wind_direction_degrees",
    "temperature_c",
    "sea_level_pressure_hpa",
).orderBy("state", "station_id", "timestamp_utc").show(20, truncate=False)

+-----------+-------------------+----+-----+---+----+-----+------+-------------+----------------------+-------------+----------------------+
|station_id |timestamp_utc      |year|month|day|hour|state|region|wind_speed_ms|wind_direction_degrees|temperature_c|sea_level_pressure_hpa|
+-----------+-------------------+----+-----+---+----+-----+------+-------------+----------------------+-------------+----------------------+
|69015093121|2018-01-01 00:56:00|2018|1    |1  |0   |CA   |West  |3.1          |180                   |15.000000    |1018.600000           |
|69015093121|2018-01-01 01:56:00|2018|1    |1  |1   |CA   |West  |2.6          |190                   |12.800000    |1019.400000           |
|69015093121|2018-01-01 02:56:00|2018|1    |1  |2   |CA   |West  |0.0          |NULL                  |11.100000    |1020.100000           |
|69015093121|2018-01-01 03:56:00|2018|1    |1  |3   |CA   |West  |0.0          |NULL                  |10.600000    |1020.900000           |
|69015093121|

In [19]:
print("Layer 5 Part C validation checklist")
print("Bronze exists in S3:", bronze_count > 0)
print("Silver exists in S3:", silver_count > 0)
print("Silver has required columns:", len(missing_columns) == 0)
print("Silver has multiple states:", silver_df.select("state").distinct().count() > 1)
print("Silver has multiple years:", silver_df.select("year").distinct().count() > 1)
print("Silver row count:", silver_count)

Layer 5 Part C validation checklist
Bronze exists in S3: True
Silver exists in S3: True
Silver has required columns: True


Silver has multiple states: True


Silver has multiple years: True
Silver row count: 28893512


# Scaled ETL Validation Summary

## Dataset Overview

- Source: NOAA ISD (Bronze → Silver pipeline)
- Years processed: **2018–2020**
- States included: **CA, TX, MN, FL**
- Total Silver rows: **28.89 million**
- Total Bronze rows: **35.50 million**
- Retention rate after parsing/cleaning/QC: **81.38%**

---

## Schema Validation

- All required columns are present:
  - station_id, timestamp_utc, date_utc
  - temporal fields (year, month, day, hour)
  - weather fields (wind, temperature, pressure, etc.)
  - metadata (state, region)
- Schema is stable and suitable for downstream analytics and ML

---

## Partitioning Validation

- Silver is partitioned by:
  - **year**
  - **state**

### Row distribution:

| Year | State | Rows |
|------|------|------|
| 2018 | CA | ~2.00M |
| 2018 | TX | ~3.67M |
| 2019 | CA | ~2.04M |
| 2020 | TX | ~3.56M |

- Balanced partitioning across states and years
- No severe skew observed

---

## Data Quality Validation

### Wind Speed Distribution

- Median: **3.1 m/s**
- 95th percentile: **7.7 m/s**
- Max: **61.3 m/s**
- Zero wind rows: **~20.6%** (expected in calm conditions)

### Null Analysis

- Critical fields (wind_speed_ms, timestamp, state, year): **0 nulls**
- Non-critical fields (pressure, visibility, ceiling): partial nulls (expected due to reporting variability)

### Physical Plausibility Checks

- No invalid values detected:
  - No negative wind speeds
  - No wind > 75 m/s
  - No invalid temperature or pressure ranges

---

## Station Coverage

- Distinct stations: **626**
- States: **4**
- Years: **3**

Distribution:

| State | Stations |
|------|--------|
| CA | 169 |
| TX | 217 |
| MN | 100 |
| FL | 140 |

---

## Performance Validation

- Query: filter (year=2020 AND state=TX)
- Rows returned: **3.56M**
- Execution time: **~1.46 seconds**

Conclusion:
- Partition pruning is working correctly
- Silver table is optimized for analytical workloads

---

## Conclusion

- Bronze and Silver layers successfully created in S3
- Silver dataset is:
  - Clean
  - Physically valid
  - Properly partitioned
  - Efficient to query
- Ready for downstream analytics and modeling